<a href="https://colab.research.google.com/github/VaggelisApostolou/Challenges-in-Detecting-Toxic-Language-in-Greek-Sports-Social-Media/blob/main/ModelComparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [ ]:
!pip install --upgrade transformers datasets accelerate

In [ ]:
import os, random, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import math
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    XLMRobertaTokenizerFast,
    XLMRobertaConfig,
    XLMRobertaForSequenceClassification,
    XLMRobertaForMaskedLM,
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, classification_report, precision_score, recall_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
XLM_CHECKPOINT = "xlm-roberta-base"
GREEKB_CHECKPOINT = "nlpaueb/bert-base-greek-uncased-v1"
MMB_CHECKPOINT = "jhu-clsp/mmBERT-base"
MASKED_XLM = "/content/drive/MyDrive/Toxic language in football Research/models/my_domain_adapted_model"
MASKED_GREEKB = "/content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT"
MASKED_MMB = "/content/drive/MyDrive/Toxic language in football Research/models/mmBERT"

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Toxic language in football Research/final_labeled_batches/final_labeled_language.csv"

# **Dataset Analysis**

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
print(df['language'].value_counts())

language
en    4999
el    1001
Name: count, dtype: int64


In [ ]:
df_en = df[df['language'] == 'en']
df_el = df[df['language'] == 'el']

In [ ]:
print(df_en['label'].value_counts())
print(df_el['label'].value_counts())

label
Non-toxic    4721
Toxic         278
Name: count, dtype: int64
label
Non-toxic    751
Toxic        250
Name: count, dtype: int64


# **Model**

## *Configuration*

In [ ]:
SEED      = 42
EPOCHS    = 8
BATCH_SZ  = 32
LR        = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_LEN   = 256
PATIENCE  = 3
GRAD_CLIP = 1.0

LOSS_TYPE = "focal"   # "ce" or "focal"
FOCAL_GAMMA = 2.0

# Toxic class emphasis
TOXIC_WEIGHT_MULT = 1.5
USE_SAMPLER = True

# Threshold objective
FBETA = 2.0              # tune threshold on F_beta (beta=2 -> recall emphasis)

set_seed(SEED)
print("Device:", device)
print("Loss:", LOSS_TYPE, "| gamma:", FOCAL_GAMMA, "| toxic_w_mult:", TOXIC_WEIGHT_MULT, "| Fbeta:", FBETA)


Device: cuda
Loss: focal | gamma: 2.0 | toxic_w_mult: 1.5 | Fbeta: 2.0


## *Load Dataset*

In [ ]:
def load_dataset(path: str):
    df = pd.read_csv(path)
    if not {"text", "label"}.issubset(df.columns):
        if df.shape[1] == 2:
            df.columns = ["text", "label"]
        else:
            raise ValueError(f"Το CSV πρέπει να έχει στήλες 'text' και 'label'. Βρέθηκαν: {df.columns.tolist()}")

    df = df.dropna(subset=["label", "text"]).copy()

    label2id = {"Non-toxic": 0, "Toxic": 1}

    if not set(df["label"].unique()).issubset(set(label2id.keys())):
        raise ValueError(f"Τα labels πρέπει να είναι {list(label2id.keys())}, βρέθηκαν: {df['label'].unique()}")

    df["label_id"] = df["label"].map(label2id)

    return df, label2id

In [ ]:
df, label2id = load_dataset(DATA_PATH)
print("Total shape:", df.shape)
print("Total label distribution:\n", df["label"].value_counts(normalize=True).round(3))
df.head()

Total shape: (6000, 5)
Total label distribution:
 label
Non-toxic    0.912
Toxic        0.088
Name: proportion, dtype: float64


,id,text,label,language,label_id
0,3438,@user @user I miss you Valbuş💛💙 Come to İstanb...,Non-toxic,en,0
1,3439,@user We're extremely glad to see y'all landed...,Non-toxic,en,0
2,3440,@user Lucky ass goal,Non-toxic,en,0
3,3441,@user Your deeds for the motherland won't go u...,Non-toxic,en,0
4,3442,@user @user Μπράβο παιδιά!!!,Non-toxic,el,0


## *Train-Test Split*

In [ ]:
# First: train (70%) vs temp (30%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
# Then: temp -> val (15%) and test (15%) i.e., split temp 50/50 stratified
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)

def pct(x): return (x.value_counts(normalize=True).rename({0:"non-toxic",1:"toxic"})*100).round(1)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist (%):\n", pct(train_df["label_id"]))
print("Val   dist (%):\n", pct(val_df["label_id"]))
print("Test  dist (%):\n", pct(test_df["label_id"]))

Split sizes: 4200 900 900
Train dist (%):
 label_id
non-toxic    91.2
toxic         8.8
Name: proportion, dtype: float64
Val   dist (%):
 label_id
non-toxic    91.2
toxic         8.8
Name: proportion, dtype: float64
Test  dist (%):
 label_id
non-toxic    91.2
toxic         8.8
Name: proportion, dtype: float64


## *Class Weights*

In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=train_df["label_id"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights[1] = class_weights[1] * TOXIC_WEIGHT_MULT  # boost toxic
class_weights = class_weights.to(device)
print("Class weights (train) with boost:", class_weights.tolist())

Class weights (train) with boost: [0.5483028888702393, 8.513513565063477]


## *Focal Loss*

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum": return loss.sum()
        return loss

if LOSS_TYPE.lower() == "focal":
    loss_fn = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA)
else:
    loss_fn = lambda logits, targets: F.cross_entropy(logits, targets, weight=class_weights)
loss_fn

FocalLoss()

# **Model 1 (XLM-RoBERTa)**

## **Classic**

In [ ]:
MODEL_NAME = XLM_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)
config = XLMRobertaConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 1.2998 | ValAcc 0.0878 | ValF1_macro 0.0807 | ValF1_toxic 0.1614 | ValPrec_toxic 0.0878 | ValRec_toxic 1.0000
Val AUROC: 0.8823
Epoch  2 | TrainLoss 0.2654 | ValAcc 0.5489 | ValF1_macro 0.4748 | ValF1_toxic 0.2776 | ValPrec_toxic 0.1615 | ValRec_toxic 0.9873
Val AUROC: 0.9072
Epoch  3 | TrainLoss 0.1749 | ValAcc 0.8633 | ValF1_macro 0.7012 | ValF1_toxic 0.4810 | ValPrec_toxic 0.3608 | ValRec_toxic 0.7215
Val AUROC: 0.9104
Epoch  4 | TrainLoss 0.1110 | ValAcc 0.8889 | ValF1_macro 0.7305 | ValF1_toxic 0.5238 | ValPrec_toxic 0.4198 | ValRec_toxic 0.6962
Val AUROC: 0.9123
Epoch  5 | TrainLoss 0.0635 | ValAcc 0.9133 | ValF1_macro 0.7544 | ValF1_toxic 0.5568 | ValPrec_toxic 0.5052 | ValRec_toxic 0.6203
Val AUROC: 0.8967
Epoch  6 | TrainLoss 0.1003 | ValAcc 0.9244 | ValF1_macro 0.7614 | ValF1_toxic 0.5641 | ValPrec_toxic 0.5714 | ValRec_toxic 0.5570
Val AUROC: 0.8953
Epoch  7 | TrainLoss 0.0480 | ValAcc 0.9089 | ValF1_macro 0.7469 | ValF1_toxic 0.5444 | ValPrec_toxic 0.48

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.97      0.92      0.95       821
           1       0.48      0.75      0.58        79

    accuracy                           0.91       900
   macro avg       0.73      0.83      0.76       900
weighted avg       0.93      0.91      0.91       900


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9233 | Val Macro-F1: 0.7771 | Val F1-toxic: 0.5965
Best thr=0.105 (beta=2.0) -> Val Acc: 0.9056 | Val Macro-F1: 0.7640 | Val F1-toxic: 0.5813
Val AUROC: 0.9163
              precision    recall  f1-score   support

           0       0.95      0.94      0.94       821
           1       0.43      0.51      0.47        79

    accuracy                           0.90       900
   macro avg       0.69      0.72      0.70       900
weighted avg       0.91      0.90      0.90       900

              precision    recall  f1-score   support

           0       0.96      0.90      0.93       821
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_XLM

### *Model Setup*

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)
config = XLMRobertaConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/197 [00:02<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/my_domain_adapted_model
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 1.1338 | ValAcc 0.0878 | ValF1_macro 0.0807 | ValF1_toxic 0.1614 | ValPrec_toxic 0.0878 | ValRec_toxic 1.0000
Val AUROC: 0.8505
Epoch  2 | TrainLoss 0.2591 | ValAcc 0.7700 | ValF1_macro 0.6353 | ValF1_toxic 0.4136 | ValPrec_toxic 0.2664 | ValRec_toxic 0.9241
Val AUROC: 0.9035
Epoch  3 | TrainLoss 0.1645 | ValAcc 0.8333 | ValF1_macro 0.6934 | ValF1_toxic 0.4863 | ValPrec_toxic 0.3333 | ValRec_toxic 0.8987
Val AUROC: 0.9137
Epoch  4 | TrainLoss 0.1404 | ValAcc 0.8511 | ValF1_macro 0.7116 | ValF1_toxic 0.5109 | ValPrec_toxic 0.3590 | ValRec_toxic 0.8861
Val AUROC: 0.9266
Epoch  5 | TrainLoss 0.1170 | ValAcc 0.9056 | ValF1_macro 0.7578 | ValF1_toxic 0.5685 | ValPrec_toxic 0.4746 | ValRec_toxic 0.7089
Val AUROC: 0.9260
Epoch  6 | TrainLoss 0.0775 | ValAcc 0.9133 | ValF1_macro 0.7639 | ValF1_toxic 0.5761 | ValPrec_toxic 0.5048 | ValRec_toxic 0.6709
Val AUROC: 0.8975
Epoch  7 | TrainLoss 0.0400 | ValAcc 0.9200 | ValF1_macro 0.7686 | ValF1_toxic 0.5814 | ValPrec_toxic 0.53

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.90      0.94       821
           1       0.43      0.78      0.56        79

    accuracy                           0.89       900
   macro avg       0.70      0.84      0.75       900
weighted avg       0.93      0.89      0.90       900


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9200 | Val Macro-F1: 0.7686 | Val F1-toxic: 0.5814
Best thr=0.048 (beta=2.0) -> Val Acc: 0.8900 | Val Macro-F1: 0.7466 | Val F1-toxic: 0.5561
Val AUROC: 0.8614
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       821
           1       0.48      0.58      0.53        79

    accuracy                           0.91       900
   macro avg       0.72      0.76      0.74       900
weighted avg       0.92      0.91      0.91       900

              precision    recall  f1-score   support

           0       0.97      0.89      0.93       821
     

# **Model 2 (Greek-BERT)**

## **Classic**

In [ ]:
MODEL_NAME = GREEKB_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

### *Data Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 0.8564 | ValAcc 0.1967 | ValF1_macro 0.1963 | ValF1_toxic 0.1793 | ValPrec_toxic 0.0985 | ValRec_toxic 1.0000
Val AUROC: 0.8413
Epoch  2 | TrainLoss 0.3198 | ValAcc 0.5989 | ValF1_macro 0.5044 | ValF1_toxic 0.2880 | ValPrec_toxic 0.1706 | ValRec_toxic 0.9241
Val AUROC: 0.8745
Epoch  3 | TrainLoss 0.1960 | ValAcc 0.7611 | ValF1_macro 0.6179 | ValF1_toxic 0.3840 | ValPrec_toxic 0.2481 | ValRec_toxic 0.8481
Val AUROC: 0.8887
Epoch  4 | TrainLoss 0.1382 | ValAcc 0.8156 | ValF1_macro 0.6588 | ValF1_toxic 0.4276 | ValPrec_toxic 0.2938 | ValRec_toxic 0.7848
Val AUROC: 0.8969
Epoch  5 | TrainLoss 0.0960 | ValAcc 0.8611 | ValF1_macro 0.6919 | ValF1_toxic 0.4635 | ValPrec_toxic 0.3506 | ValRec_toxic 0.6835
Val AUROC: 0.8840
Epoch  6 | TrainLoss 0.0803 | ValAcc 0.8644 | ValF1_macro 0.6937 | ValF1_toxic 0.4649 | ValPrec_toxic 0.3557 | ValRec_toxic 0.6709
Val AUROC: 0.8822
Epoch  7 | TrainLoss 0.0687 | ValAcc 0.8633 | ValF1_macro 0.6900 | ValF1_toxic 0.4581 | ValPrec_toxic 0.35

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.97      0.86      0.91       821
           1       0.33      0.75      0.46        79

    accuracy                           0.85       900
   macro avg       0.65      0.80      0.69       900
weighted avg       0.92      0.85      0.87       900


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.8833 | Val Macro-F1: 0.6981 | Val F1-toxic: 0.4615
Best thr=0.091 (beta=2.0) -> Val Acc: 0.8467 | Val Macro-F1: 0.6858 | Val F1-toxic: 0.4609
Val AUROC: 0.8755
              precision    recall  f1-score   support

           0       0.96      0.91      0.93       821
           1       0.40      0.66      0.50        79

    accuracy                           0.88       900
   macro avg       0.68      0.78      0.72       900
weighted avg       0.92      0.88      0.90       900

              precision    recall  f1-score   support

           0       0.98      0.84      0.90       821
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_GREEKB

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

### *Data Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 0.6965 | ValAcc 0.2322 | ValF1_macro 0.2297 | ValF1_toxic 0.1861 | ValPrec_toxic 0.1026 | ValRec_toxic 1.0000
Val AUROC: 0.8347
Epoch  2 | TrainLoss 0.2965 | ValAcc 0.5989 | ValF1_macro 0.4983 | ValF1_toxic 0.2736 | ValPrec_toxic 0.1627 | ValRec_toxic 0.8608
Val AUROC: 0.8010
Epoch  3 | TrainLoss 0.2093 | ValAcc 0.6778 | ValF1_macro 0.5388 | ValF1_toxic 0.2857 | ValPrec_toxic 0.1774 | ValRec_toxic 0.7342
Val AUROC: 0.7912
Epoch  4 | TrainLoss 0.1738 | ValAcc 0.7200 | ValF1_macro 0.5679 | ValF1_toxic 0.3115 | ValPrec_toxic 0.1986 | ValRec_toxic 0.7215
Val AUROC: 0.8026
Epoch  5 | TrainLoss 0.1522 | ValAcc 0.7233 | ValF1_macro 0.5536 | ValF1_toxic 0.2783 | ValPrec_toxic 0.1805 | ValRec_toxic 0.6076
Val AUROC: 0.7792
Epoch  6 | TrainLoss 0.1264 | ValAcc 0.7056 | ValF1_macro 0.5519 | ValF1_toxic 0.2895 | ValPrec_toxic 0.1837 | ValRec_toxic 0.6835
Val AUROC: 0.8036
Epoch  7 | TrainLoss 0.1113 | ValAcc 0.7456 | ValF1_macro 0.5701 | ValF1_toxic 0.2954 | ValPrec_toxic 0.19

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.64      0.78       821
           1       0.19      0.85      0.30        79

    accuracy                           0.66       900
   macro avg       0.58      0.75      0.54       900
weighted avg       0.91      0.66      0.73       900


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.7456 | Val Macro-F1: 0.5701 | Val F1-toxic: 0.2954
Best thr=0.034 (beta=2.0) -> Val Acc: 0.6600 | Val Macro-F1: 0.5398 | Val F1-toxic: 0.3045
Val AUROC: 0.7799
              precision    recall  f1-score   support

           0       0.96      0.74      0.83       821
           1       0.19      0.65      0.29        79

    accuracy                           0.73       900
   macro avg       0.57      0.69      0.56       900
weighted avg       0.89      0.73      0.78       900

              precision    recall  f1-score   support

           0       0.97      0.64      0.77       821
     

# **Model 3 (mmBERT)**

## **Classic**

In [ ]:
MODEL_NAME = MMB_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 0.6447 | ValAcc 0.6756 | ValF1_macro 0.5622 | ValF1_toxic 0.3394 | ValPrec_toxic 0.2066 | ValRec_toxic 0.9494
Val AUROC: 0.9067
Epoch  2 | TrainLoss 0.1113 | ValAcc 0.7567 | ValF1_macro 0.6237 | ValF1_toxic 0.4000 | ValPrec_toxic 0.2552 | ValRec_toxic 0.9241
Val AUROC: 0.9233
Epoch  3 | TrainLoss 0.0339 | ValAcc 0.8856 | ValF1_macro 0.7301 | ValF1_toxic 0.5253 | ValPrec_toxic 0.4130 | ValRec_toxic 0.7215
Val AUROC: 0.9051
Epoch  4 | TrainLoss 0.0206 | ValAcc 0.9222 | ValF1_macro 0.7289 | ValF1_toxic 0.5000 | ValPrec_toxic 0.5738 | ValRec_toxic 0.4430
Val AUROC: 0.9116
Epoch  5 | TrainLoss 0.0026 | ValAcc 0.9256 | ValF1_macro 0.7242 | ValF1_toxic 0.4885 | ValPrec_toxic 0.6154 | ValRec_toxic 0.4051
Val AUROC: 0.9020
Epoch  6 | TrainLoss 0.0032 | ValAcc 0.9256 | ValF1_macro 0.6938 | ValF1_toxic 0.4274 | ValPrec_toxic 0.6579 | ValRec_toxic 0.3165
Val AUROC: 0.8940
Epoch  7 | TrainLoss 0.0000 | ValAcc 0.9256 | ValF1_macro 0.7032 | ValF1_toxic 0.4463 | ValPrec_toxic 0.64

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.87      0.92       821
           1       0.37      0.81      0.51        79

    accuracy                           0.86       900
   macro avg       0.67      0.84      0.71       900
weighted avg       0.93      0.86      0.88       900


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.8856 | Val Macro-F1: 0.7301 | Val F1-toxic: 0.5253
Best thr=0.195 (beta=2.0) -> Val Acc: 0.8611 | Val Macro-F1: 0.7126 | Val F1-toxic: 0.5059
Val AUROC: 0.9051
              precision    recall  f1-score   support

           0       0.98      0.90      0.94       821
           1       0.44      0.78      0.56        79

    accuracy                           0.89       900
   macro avg       0.71      0.84      0.75       900
weighted avg       0.93      0.89      0.91       900

              precision    recall  f1-score   support

           0       0.98      0.86      0.92       821
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_MMB

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/mmBERT
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 0.7184 | ValAcc 0.6267 | ValF1_macro 0.5265 | ValF1_toxic 0.3086 | ValPrec_toxic 0.1843 | ValRec_toxic 0.9494
Val AUROC: 0.8958
Epoch  2 | TrainLoss 0.1212 | ValAcc 0.8356 | ValF1_macro 0.6924 | ValF1_toxic 0.4825 | ValPrec_toxic 0.3333 | ValRec_toxic 0.8734
Val AUROC: 0.9268
Epoch  3 | TrainLoss 0.0283 | ValAcc 0.9044 | ValF1_macro 0.7560 | ValF1_toxic 0.5657 | ValPrec_toxic 0.4706 | ValRec_toxic 0.7089
Val AUROC: 0.9220
Epoch  4 | TrainLoss 0.0092 | ValAcc 0.9289 | ValF1_macro 0.7615 | ValF1_toxic 0.5616 | ValPrec_toxic 0.6119 | ValRec_toxic 0.5190
Val AUROC: 0.9108
Epoch  5 | TrainLoss 0.0038 | ValAcc 0.9333 | ValF1_macro 0.7736 | ValF1_toxic 0.5833 | ValPrec_toxic 0.6462 | ValRec_toxic 0.5316
Val AUROC: 0.9161
Epoch  6 | TrainLoss 0.0012 | ValAcc 0.9311 | ValF1_macro 0.7430 | ValF1_toxic 0.5231 | ValPrec_toxic 0.6667 | ValRec_toxic 0.4304
Val AUROC: 0.9112
Epoch  7 | TrainLoss 0.0007 | ValAcc 0.9333 | ValF1_macro 0.7362 | ValF1_toxic 0.5082 | ValPrec_toxic 0.72

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.91      0.94       821
           1       0.46      0.76      0.57        79

    accuracy                           0.90       900
   macro avg       0.72      0.84      0.76       900
weighted avg       0.93      0.90      0.91       900


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9333 | Val Macro-F1: 0.7736 | Val F1-toxic: 0.5833
Best thr=0.003 (beta=2.0) -> Val Acc: 0.9000 | Val Macro-F1: 0.7574 | Val F1-toxic: 0.5714
Val AUROC: 0.9161
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       821
           1       0.64      0.53      0.58        79

    accuracy                           0.93       900
   macro avg       0.80      0.75      0.77       900
weighted avg       0.93      0.93      0.93       900

              precision    recall  f1-score   support

           0       0.97      0.88      0.92       821
     

# **GreekBert special greek dataset case**

## *Configuration*

In [ ]:
SEED      = 42
EPOCHS    = 8
BATCH_SZ  = 32
LR        = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_LEN   = 256
PATIENCE  = 3
GRAD_CLIP = 1.0

LOSS_TYPE = "focal"   # "ce" or "focal"
FOCAL_GAMMA = 2.0

# Toxic class emphasis
TOXIC_WEIGHT_MULT = 1.5
USE_SAMPLER = True

# Threshold objective
FBETA = 2.0              # tune threshold on F_beta (beta=2 -> recall emphasis)

set_seed(SEED)
print("Device:", device)
print("Loss:", LOSS_TYPE, "| gamma:", FOCAL_GAMMA, "| toxic_w_mult:", TOXIC_WEIGHT_MULT, "| Fbeta:", FBETA)


Device: cuda
Loss: focal | gamma: 2.0 | toxic_w_mult: 1.5 | Fbeta: 2.0


## *Load Dataset*

In [ ]:
def load_dataset():
    df = df_el
    if not {"text", "label"}.issubset(df.columns):
        if df.shape[1] == 2:
            df.columns = ["text", "label"]
        else:
            raise ValueError(f"Το CSV πρέπει να έχει στήλες 'text' και 'label'. Βρέθηκαν: {df.columns.tolist()}")

    df = df.dropna(subset=["label", "text"]).copy()

    label2id = {"Non-toxic": 0, "Toxic": 1}

    if not set(df["label"].unique()).issubset(set(label2id.keys())):
        raise ValueError(f"Τα labels πρέπει να είναι {list(label2id.keys())}, βρέθηκαν: {df['label'].unique()}")

    df["label_id"] = df["label"].map(label2id)

    return df, label2id

In [ ]:
df, label2id = load_dataset()
print("Total shape:", df.shape)
print("Total label distribution:\n", df["label"].value_counts(normalize=True).round(3))
df.head()

Total shape: (1001, 5)
Total label distribution:
 label
Non-toxic    0.75
Toxic        0.25
Name: proportion, dtype: float64


,id,text,label,language,label_id
4,3442,@user @user Μπράβο παιδιά!!!,Non-toxic,el,0
9,3447,@user 200 ΚΙΛΆ ΤΑ ΈΧΕΙΣ ΜΩΡΗ ΑΡΡΩΣΤΙΑ,Toxic,el,1
14,3452,@user @user γαμα τους Φίλιππε ☘️,Toxic,el,1
18,3456,@user ΤΙ ΕΒΑΛΕΣ ΡΕ ΚΑΡΓΙΟΛΗ; 💦💦💦💦,Toxic,el,1
19,3457,@user @user Απαν κατα απαν,Non-toxic,el,0


## *Train-Test Split*

In [ ]:
# First: train (70%) vs temp (30%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
# Then: temp -> val (15%) and test (15%) i.e., split temp 50/50 stratified
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)

def pct(x): return (x.value_counts(normalize=True).rename({0:"non-toxic",1:"toxic"})*100).round(1)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist (%):\n", pct(train_df["label_id"]))
print("Val   dist (%):\n", pct(val_df["label_id"]))
print("Test  dist (%):\n", pct(test_df["label_id"]))

Split sizes: 700 150 151
Train dist (%):
 label_id
non-toxic    75.0
toxic        25.0
Name: proportion, dtype: float64
Val   dist (%):
 label_id
non-toxic    75.3
toxic        24.7
Name: proportion, dtype: float64
Test  dist (%):
 label_id
non-toxic    74.8
toxic        25.2
Name: proportion, dtype: float64


## *Class Weights*

In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=train_df["label_id"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights[1] = class_weights[1] * TOXIC_WEIGHT_MULT  # boost toxic
class_weights = class_weights.to(device)
print("Class weights (train) with boost:", class_weights.tolist())

Class weights (train) with boost: [0.6666666865348816, 3.0]


## *Focal Loss*

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum": return loss.sum()
        return loss

if LOSS_TYPE.lower() == "focal":
    loss_fn = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA)
else:
    loss_fn = lambda logits, targets: F.cross_entropy(logits, targets, weight=class_weights)
loss_fn

FocalLoss()

## **Classic**

In [ ]:
MODEL_NAME = GREEKB_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	tho

### *Data Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 0.4327 | ValAcc 0.2467 | ValF1_macro 0.1979 | ValF1_toxic 0.3957 | ValPrec_toxic 0.2467 | ValRec_toxic 1.0000
Val AUROC: 0.5810
Epoch  2 | TrainLoss 0.2918 | ValAcc 0.2467 | ValF1_macro 0.1979 | ValF1_toxic 0.3957 | ValPrec_toxic 0.2467 | ValRec_toxic 1.0000
Val AUROC: 0.7228
Epoch  3 | TrainLoss 0.2695 | ValAcc 0.2467 | ValF1_macro 0.1979 | ValF1_toxic 0.3957 | ValPrec_toxic 0.2467 | ValRec_toxic 1.0000
Val AUROC: 0.8433
Epoch  4 | TrainLoss 0.2361 | ValAcc 0.3200 | ValF1_macro 0.2989 | ValF1_toxic 0.4205 | ValPrec_toxic 0.2662 | ValRec_toxic 1.0000
Val AUROC: 0.8309
Epoch  5 | TrainLoss 0.2247 | ValAcc 0.3733 | ValF1_macro 0.3661 | ValF1_toxic 0.4337 | ValPrec_toxic 0.2791 | ValRec_toxic 0.9730
Val AUROC: 0.8620
Epoch  6 | TrainLoss 0.2007 | ValAcc 0.4733 | ValF1_macro 0.4733 | ValF1_toxic 0.4768 | ValPrec_toxic 0.3158 | ValRec_toxic 0.9730
Val AUROC: 0.8699
Epoch  7 | TrainLoss 0.1767 | ValAcc 0.6200 | ValF1_macro 0.6109 | ValF1_toxic 0.5512 | ValPrec_toxic 0.38

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.96      0.60      0.74       113
           1       0.43      0.92      0.59        37

    accuracy                           0.68       150
   macro avg       0.69      0.76      0.66       150
weighted avg       0.83      0.68      0.70       150


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.6200 | Val Macro-F1: 0.6109 | Val F1-toxic: 0.5512
Best thr=0.537 (beta=2.0) -> Val Acc: 0.6800 | Val Macro-F1: 0.6627 | Val F1-toxic: 0.5862
Val AUROC: 0.8644
              precision    recall  f1-score   support

           0       0.92      0.43      0.59       113
           1       0.35      0.89      0.50        38

    accuracy                           0.55       151
   macro avg       0.64      0.66      0.55       151
weighted avg       0.78      0.55      0.57       151

              precision    recall  f1-score   support

           0       0.90      0.53      0.67       113
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_GREEKB

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if

### *Data Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 0.4006 | ValAcc 0.2467 | ValF1_macro 0.1979 | ValF1_toxic 0.3957 | ValPrec_toxic 0.2467 | ValRec_toxic 1.0000
Val AUROC: 0.6604
Epoch  2 | TrainLoss 0.2939 | ValAcc 0.2800 | ValF1_macro 0.2457 | ValF1_toxic 0.4066 | ValPrec_toxic 0.2552 | ValRec_toxic 1.0000
Val AUROC: 0.7120
Epoch  3 | TrainLoss 0.2687 | ValAcc 0.3067 | ValF1_macro 0.2816 | ValF1_toxic 0.4157 | ValPrec_toxic 0.2624 | ValRec_toxic 1.0000
Val AUROC: 0.6687
Epoch  4 | TrainLoss 0.2790 | ValAcc 0.3200 | ValF1_macro 0.2989 | ValF1_toxic 0.4205 | ValPrec_toxic 0.2662 | ValRec_toxic 1.0000
Val AUROC: 0.7259
Epoch  5 | TrainLoss 0.2466 | ValAcc 0.3400 | ValF1_macro 0.3292 | ValF1_toxic 0.4142 | ValPrec_toxic 0.2652 | ValRec_toxic 0.9459
Val AUROC: 0.7470
Epoch  6 | TrainLoss 0.2424 | ValAcc 0.3867 | ValF1_macro 0.3839 | ValF1_toxic 0.4250 | ValPrec_toxic 0.2764 | ValRec_toxic 0.9189
Val AUROC: 0.7245
Epoch  7 | TrainLoss 0.2309 | ValAcc 0.3400 | ValF1_macro 0.3292 | ValF1_toxic 0.4142 | ValPrec_toxic 0.26

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.90      0.58      0.70       113
           1       0.38      0.81      0.52        37

    accuracy                           0.63       150
   macro avg       0.64      0.69      0.61       150
weighted avg       0.77      0.63      0.66       150


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.3867 | Val Macro-F1: 0.3839 | Val F1-toxic: 0.4250
Best thr=0.677 (beta=2.0) -> Val Acc: 0.6333 | Val Macro-F1: 0.6122 | Val F1-toxic: 0.5217
Val AUROC: 0.7245
              precision    recall  f1-score   support

           0       0.89      0.22      0.35       113
           1       0.28      0.92      0.43        38

    accuracy                           0.40       151
   macro avg       0.59      0.57      0.39       151
weighted avg       0.74      0.40      0.37       151

              precision    recall  f1-score   support

           0       0.84      0.57      0.68       113
     